# CNW-Grass gas exchange for developers, 0: overview

**For:** developers who want to read, check, change or couple the gas exchange model. **You need:** the user tutorial
`../user/01_first_run.ipynb` (what the model computes), Python, numpy. **Run time:** a few seconds.

This series:

| | notebook | what it covers |
|---|---|---|
| 0 | `00_overview.ipynb` | the modules, the original CNW-Grass gas exchange, how the port is validated, the corrections |
| 1 | `01_gas_exchange_in_metafspm.ipynb` | metafspm's component formalism on `GasExchange` (declarations, steps, forcings, settings), the equations on one element, the stomatal models, the corrections, `GasExchange` in a plant model and a scene with Caribu's light, its transpiration into MECHA |
| 2 | `02_extend_the_component.ipynb` | add a variable and a step to the component, run it in a scene, test it |

## 1. Where the model comes from

CNW-Grass (`openalea.cnwgrass`) is a functional–structural model of grasses' carbon and nitrogen metabolism, growth
and morphogenesis. Its submodule `openalea.cnwgrass.gasexchange` (the **legacy** here, kept
as it is) computes each shoot element's photosynthesis (Farquhar), stomatal conductance, surface temperature and
transpiration, from dicts of elements' inputs, its parameters module-level globals.

`openalea.cnwgrass.metafspm` holds CNW-Grass's submodules ported as metafspm components. The gas exchange is the
first; the others (morphogenesis, growth, CN metabolism, respiration, senescence, hydraulics) are not ported yet.

In [1]:
import importlib
import pkgutil

import openalea.cnwgrass.metafspm as package

print(package.__doc__.strip(), "\n")
for module in pkgutil.iter_modules(package.__path__):
    doc = (importlib.import_module(f"openalea.cnwgrass.metafspm.{module.name}").__doc__ or "").strip()
    print(f"{module.name:14} {doc.splitlines()[0]}")

CNW-Grass's submodules as metafspm components (openalea.metafspm), one module each:

    gas_exchange    GasExchange: each shoot element's photosynthesis (Farquhar), stomatal conductance, surface
                    temperature and transpiration (the legacy openalea.cnwgrass.gasexchange)

The other submodules (morphogenesis, growth, CN metabolism, respiration, senescence, hydraulics) are not ported yet. 



gas_exchange   CNW-Grass's gas exchange as a metafspm component: each shoot element's photosynthesis, stomatal conductance, surface
simulation     CNW-Grass's gas exchange for users: a small wheat stand's photosynthesis and transpiration in a few lines, its


- `gas_exchange.py` is the component `GasExchange`, the equations it calls for one element (`element_exchange`,
  `photosynthesis`, `stomatal_conductance`, `organ_temperature`, ...), the corrections (`CORRECTIONS`), and the
  helpers a plant model uses: `adel_links()` (the links from Adel's architecture), `surfacic_photosynthetic_nitrogen`,
  `surfacic_total_nitrogen`, `surfacic_nonstructural_carbohydrates` (the inputs from a CN model's amounts, as the
  legacy computes them), `TOTAL_NITROGEN` (the legacy's other nitrogen parametrisation).
- `simulation.py` is the users' entry point: `simulate`, a wheat stand of Adel's plants lit by Caribu, its tables; its
  plant model `ExchangingWheat`.

The equations are in the module's docstring, rendered in the documentation's page `doc/metafspm/gas_exchange.rst`
with the component's declarations, the corrections, and every behaviour of the legacy with what the port does.

## 2. How the port is validated

The tests, in `test/`, run with `pytest test/test_gas_exchange.py test/test_simulation.py` from the repository's root:

In [2]:
import ast
from pathlib import Path

for name in ("test_gas_exchange.py", "test_simulation.py"):
    tree = ast.parse(Path("../../../test", name).read_text())
    print(name, "\n   ", " ".join((ast.get_docstring(tree) or "").split()))
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name.startswith("test_"):
            print("     ", node.name)

test_gas_exchange.py 
    GasExchange (openalea.cnwgrass.metafspm.gas_exchange) against the legacy openalea.cnwgrass.gasexchange Simulation: its equations element by element, bit for bit with the legacy's numerics (corrections=()), for every stomatal model and option of the legacy; the component on Adel's plants (ShootArchitecture) in a metafspm Scene, bit for bit on every element it computes, its parameters per plant; the corrections and the fixed failures.
      test_the_equations_reproduce_the_legacy
      test_the_legacy_versions_reproduced
      test_the_primitive_scale
      test_the_standalone_example
      test_the_corrections
      test_the_legacy_failures_fixed
      test_the_component_reproduces_the_legacy
      test_the_component_declarations_and_the_organs
      test_the_component_restarts_Ci
      test_the_tuzet_feedback
      test_the_component_at_the_primitive_scale
test_simulation.py 
    The users' entry point (openalea.cnwgrass.metafspm.simulation): a small wheat sta

Three levels:

- **Exactness.** With `corrections=()`, the ported equations reproduce the legacy `Simulation` bit for bit on random
  elements, for the four stomatal models and three weathers, with or without the carbohydrates' inhibition, with
  either nitrogen parametrisation, at the primitive scale and on the legacy's standalone example; the component does
  too, on Adel's plants in a metafspm scene, two plants with their own parameters.
- **Corrections.** Each behaviour of the legacy found to be an error is corrected by a named correction, tested by
  what it changes (`test_the_corrections`); `"recommended"`, the component's default, applies them all. The
  legacy's failures (divisions by zero) are fixed without a name (`test_the_legacy_failures_fixed`).
- **The users' entry point** is tested against the elements' variables it sums (`test_simulation.py`).

The corrections, by name:

In [3]:
from openalea.cnwgrass.metafspm.gas_exchange import NAMES

for name, description in NAMES.items():
    print(f"{name:>25}  {description}")

    vapour_pressure_slope  the slope of the saturation vapour pressure at the air temperature, Tetens' derivative es(Ta) 17.4 · 239 / (Ta + 239)² with Ta in °C (the legacy adds 273.15 to Ta: a slope about four times too small at the fixed point's first iteration)
 compensation_point_units  the CO2 compensation point Γ of the Leuning, Tuzet and hydraulics models in µmol mol-1, as Cs, its temperature in K (the legacy: Γ0 in mol mol-1 and Ta in °C minus Tref' in K, Γ about 0.002 µmol mol-1 instead of about 27)
    primitive_aggregation  at the primitive scale, every output the area-weighted mean of the primitives' (the legacy: Ag only, the others the last primitive's)


## 3. What to read next

- `01_gas_exchange_in_metafspm.ipynb` reads the component's code through metafspm's component formalism, then the
  equations on one element, the stomatal models and the corrections, and couples the component in a plant model,
  in a scene with Caribu's light, and to MECHA.
- `02_extend_the_component.ipynb` adds a variable and a step, and tests them.
- metafspm's own tutorial (`examples/soil_plant_atmosphere/tutorial_fspm2026_general_usage.ipynb` in the metafspm
  repository) builds components and couples them step by step; its user guide, `docs/user.md`, describes every part
  of the framework.